# Visualize SCALP-lite Embedding

Set `SCALP_INPUT_H5AD` to your input file. Optionally set `SCALP_OUTPUT_H5AD` for the annotated output.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from scalp_lite import ScalpEstimator, save_h5ad, plot_embedding_pair


def resolve_path(env_name, default):
    path = Path(os.environ.get(env_name, default))
    if path.exists() or path.is_absolute():
        return path
    notebook_relative = Path("..") / path
    return notebook_relative if notebook_relative.exists() else path


input_path = resolve_path("SCALP_INPUT_H5AD", "data/cellrank-pancreas.h5ad")
output_path = Path(os.environ["SCALP_OUTPUT_H5AD"]) if "SCALP_OUTPUT_H5AD" in os.environ else input_path.with_name(f"{input_path.stem}-scalp.h5ad")
batch_key = os.environ.get("SCALP_BATCH_KEY", "batch")
label_key = os.environ.get("SCALP_LABEL_KEY", "label")

estimator = ScalpEstimator(batch_key=batch_key, label_key=label_key)


In [ ]:
adata = estimator.input(input_path)

if batch_key not in adata.obs:
    # The bundled development dataset has no true batch, so split cells deterministically for smoke testing.
    adata.obs[batch_key] = pd.Categorical([f"split_{i % 3}" for i in range(adata.n_obs)])

if label_key not in adata.obs:
    for candidate in ("clusters_fine", "clusters", "clusters_coarse", "cell_type", "celltype", "leiden", "louvain"):
        if candidate in adata.obs:
            adata.obs[label_key] = adata.obs[candidate].astype("category")
            break

adata = estimator.preprocess(adata, n_top_genes=2000, max_cells=None)
adata


In [ ]:
graph = estimator.data_to_graph(adata)
adata.obsm["X_scalp"] = estimator.graph_to_vector(graph)


In [ ]:
plot_embedding_pair(adata, embedding_key="X_scalp", batch_key=batch_key, label_key=label_key);


In [ ]:
save_h5ad(adata, output_path)
output_path